# Stage 3 — Eval / Compare (Colab)

On the held-out eval set (never used to fit `W` or train the student), computes three embeddings and clusters all of them:
- **full** — raw CLIP embedding (uncompressed baseline)
- **oracle_<method>** — CLIP embedding → `W` (upper bound, needs foundation-model access)
- **student** — trained `SmallCNN(image)` directly (no CLIP needed at inference)
- **random_init** — same architecture, zero training (sanity floor)

`retained_gain` = fraction of oracle's improvement-over-full that the student preserves. Threshold from `distillation_experiment_prompt.md`: ≥0.90 average across the four metrics.

**Prerequisite**: `stage1_oracle_colab.ipynb` and `stage2_student_colab.ipynb` have both run — this notebook only reads their saved outputs, no CLIP forward pass or training happens here.

Run on a Colab GPU/CPU runtime (student inference is cheap; a CPU runtime works fine).

In [1]:
import os

REPO_URL = "https://github.com/Nizaxga/distillation-embedding-experiment.git"
REPO_DIR = "/content/distillation-embedding-experiment"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

/content/distillation-embedding-experiment


In [ ]:
!pip install -q -r requirements.txt

## Reconnect to Stage 1 + Stage 2 outputs

Same `save_dir`/symlink as the previous two notebooks — must match exactly, or `outputs/embeddings/`, `outputs/compression/<run_name>/`, and `outputs/checkpoints/<run_name>/` won't be visible here.

In [2]:
from google.colab import drive

drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/representation-learning"
print(f"[LOG] save_dir={save_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[LOG] save_dir=/content/drive/MyDrive/representation-learning


In [3]:
os.environ["IMAGENET_DOG15_CACHE"] = os.path.join(save_dir, ".cache/imagenet-dog-15")

In [4]:
DRIVE_OUTPUTS = os.path.join(save_dir, "output-distillation-embedding-experiment")

if not os.path.exists("outputs"):
    os.makedirs(DRIVE_OUTPUTS, exist_ok=True)
    os.symlink(DRIVE_OUTPUTS, "outputs")

In [5]:
import sys

sys.path.insert(0, REPO_DIR)

import numpy as np
import torch
from PIL import Image

from src.compress.fit import compression_dir
from src.data.imagenet_dog15 import load_dog15, split_fit_eval
from src.device import get_device
from src.embeddings.cache import load_embeddings
from src.eval.clustering import cluster_and_score, retained_gain
from src.eval.efficiency import cpu_latency_ms, flops, model_size_mb, param_count
from src.students.models import build_student
from src.students.train import checkpoint_dir, student_input_transform
from src.utils.config import load_config

In [6]:
cfg = load_config("configs/pilot_dog15_clip_pca.yaml")
device = get_device()
print(f"run_name={cfg.run_name}")

[device] using CUDA: Tesla T4
run_name=imagenet_dog15_clip_vit_b32_pca_k32_m1000_small_cnn_seed0


## 1. Rebuild the deterministic split, load cached full + oracle eval embeddings

In [7]:
records = load_dog15(seed=cfg.seed)
fit_records, eval_records = split_fit_eval(records, cfg.m, seed=cfg.seed)
eval_ids = [r.image_id for r in eval_records]
eval_labels = np.array([r.label for r in eval_records])
n_clusters = len(set(r.label for r in records))

eval_full = load_embeddings(cfg.dataset, cfg.backbone, eval_ids)  # from Stage 1's CLIP cache
eval_oracle = np.load(compression_dir(cfg.run_name) / "eval_oracle.npy")  # from Stage 1
print(f"eval_full={eval_full.shape}  eval_oracle={eval_oracle.shape}")

[data] label dirs under /content/drive/MyDrive/representation-learning/.cache/imagenet-dog-15 aren't synset ids (['0', '1', '10', '11', '12', '13', '14', '2', '3', '4', '5', '6', '7', '8', '9']); assigning class indices by sort order.
[data] loaded 1500 ImageNet-Dog-15 images from precached dir /content/drive/MyDrive/representation-learning/.cache/imagenet-dog-15
eval_full=(500, 512)  eval_oracle=(500, 32)


## 2. Embed the eval set with the trained student + the random-init floor

Both checkpoints came from Stage 2 (`outputs/checkpoints/<run_name>/final.pt` and `random_init.pt`) — no training happens here, just a forward pass.

In [16]:
ckpt_dir = checkpoint_dir(cfg.run_name)

student_model = build_student(cfg.student_arch, cfg.k)
student_model.load_state_dict(torch.load(ckpt_dir / "final.pt", map_location="cpu"))

random_model = build_student(cfg.student_arch, cfg.k)
random_model.load_state_dict(torch.load(ckpt_dir / "random_init.pt", map_location="cpu"))

tfm = student_input_transform()


def embed_with(m: torch.nn.Module) -> np.ndarray:
    m = m.to(device).eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(eval_records), 64):
            batch = eval_records[i : i + 64]
            imgs = torch.stack(
                [tfm(Image.open(r.path).convert("RGB")) for r in batch]
            ).to(device)
            out.append(m(imgs).cpu().numpy())
    return np.concatenate(out)


eval_student = embed_with(student_model)
eval_random = embed_with(random_model)
print(f"eval_student={eval_student.shape}  eval_random={eval_random.shape}")

eval_student=(500, 32)  eval_random=(500, 32)


## 3. Cluster + score all four rows

In [15]:
results = {
    "full": cluster_and_score(eval_full, eval_labels, n_clusters, cfg.eval_seeds),
    f"oracle_{cfg.method}": cluster_and_score(eval_oracle, eval_labels, n_clusters, cfg.eval_seeds),
    "student": cluster_and_score(eval_student, eval_labels, n_clusters, cfg.eval_seeds),
    "random_init": cluster_and_score(eval_random, eval_labels, n_clusters, cfg.eval_seeds),
}

metrics = ["v_measure", "nmi", "ari", "acc"]
print(f"{'row':<14}" + "".join(f"{m:>12}" for m in metrics))
for row_name, scores in results.items():
    print(f"{row_name:<14}" + "".join(f"{scores[m]:>12.4f}" for m in metrics))

# row              v_measure         nmi         ari         acc
# full                0.6060      0.6060      0.4090      0.5840
# oracle_pca          0.6491      0.6491      0.4833      0.6520
# student             0.2026      0.2026      0.0558      0.2060
# random_init         0.1197      0.1197      0.0145      0.1680

# row              v_measure         nmi         ari         acc
# full                0.6060      0.6060      0.4090      0.5840
# oracle_pca          0.6491      0.6491      0.4833      0.6520
# student             0.1895      0.1895      0.0439      0.2060
# random_init         0.1197      0.1197      0.0145      0.1680

row              v_measure         nmi         ari         acc
full                0.6060      0.6060      0.4090      0.5840
oracle_pca          0.6491      0.6491      0.4833      0.6520
student             0.1895      0.1895      0.0439      0.2060
random_init         0.1197      0.1197      0.0145      0.1680


## 4. Retained gain

Near 1.0 = student preserves oracle's full improvement over `full`. Near 0 = student is no better than uncompressed. Negative = student is worse than not compressing at all. Pre-registered threshold: ≥0.90 average.

In [10]:
gains = {}
for metric in metrics:
    gains[metric] = retained_gain(
        results["student"][metric], results["full"][metric], results[f"oracle_{cfg.method}"][metric]
    )
    print(f"  {metric}: {gains[metric]:.4f}")
print(f"  average: {np.mean(list(gains.values())):.4f}")

  v_measure: -9.3620
  nmi: -9.3620
  ari: -4.7511
  acc: -5.5588
  average: -7.2585


## 5. Efficiency (student, edge-deployment numbers)

In [11]:
print(f"params: {param_count(student_model):,}")
print(f"size fp32: {model_size_mb(student_model):.2f} MB")
print(f"size int8: {model_size_mb(student_model, quantized=True):.2f} MB")
print(f"cpu latency: {cpu_latency_ms(student_model):.2f} ms/image")
try:
    print(f"flops: {flops(student_model)/1e6:.2f} MFLOPs")
except Exception as e:
    print(f"flops: failed ({e})")

params: 397,600
size fp32: 1.52 MB
size int8: 1.49 MB
cpu latency: 6.78 ms/image
flops: 374.63 MFLOPs


## Done

This is the full pipeline result for one config (`configs/pilot_dog15_clip_pca.yaml`). If `retained_gain` average is well under 0.90, the student needs more training (see Stage 2 notebook's fidelity numbers) before this result means anything — a low-fidelity student invalidates the comparison, it doesn't falsify the paper's oracle claim.